# The manuscript's figures

This notebook runs every figure generator of `figures/` in order and writes the 81 PDF panels the
manuscript includes. Sixteen of the 22 figures draw from data committed in this repository: the diagrams, the validation
figures (values in `figures/data/figure_data/`), the acquisition rounds, the closing comparison and the datasets
by melting regime. The other six need FLOW-3D ground truth: five show it beside a prediction (`fig:rp-artifact`, `fig:varspeed-pred`,
`fig:multibeam-pred`, `fig:preheat-pred`, `fig:twoalloy-pred`), and `fig:gaot-smoothing` reuses the frames of the first. Their predictions are committed. Their ground truth is
FLOW-3D WELD output, distributed only in the restricted record (doi:10.5281/zenodo.23325088), and is rebuilt from it
here.

**What you need**

* Python 3.10 with numpy, matplotlib, Pillow, pyvista (off-screen) and jupyter, set up as in `reproducibility/README.md`.
  `pdftoppm` (poppler) for the comparison at the end.
* For the six ground-truth figures, `LPNO_RESTRICTED`: a folder holding the restricted record, with the payload tars
  rejoined (`cat <name>.part?? > <name>`) and `restricted_files_main.tar`. Fig 10 also needs an AiiDA profile holding
  the open archives and three restored payloads (cell 4). Without these, the notebook skips those figures and says so.
* Optionally `LPNO_PUBLISHED_FIGS`: a folder with the published figure PDFs (the `Figs/` folder of the manuscript's
  source), against which every regenerated panel is compared.

Figures are written to `LPNO_FIG_OUT` (default `reproducibility/figures_out/`), never over anything else.

In [1]:
import csv
import json
import os
import subprocess
import sys
from pathlib import Path

HERE = Path.cwd()
CODE = HERE.parent
FIGNB = HERE / "figures"
OUT = Path(os.environ.get("LPNO_FIG_OUT", HERE / "figures_out")).resolve()
_default_restricted = HERE.parent / "data_deposit" / "restricted_record"   # where data_deposit/scripts/download.py puts it
RESTRICTED = os.environ.get("LPNO_RESTRICTED") or (str(_default_restricted) if any(_default_restricted.glob("payloads_*.tar")) else None)
PUBLISHED = os.environ.get("LPNO_PUBLISHED_FIGS")
OUT.mkdir(parents=True, exist_ok=True)
ENV = {**os.environ, "LPNO_FIG_DIR": str(OUT), "LPNO_REPLACE_FIGURES": "1", "PYVISTA_OFF_SCREEN": "true",
       "PYTHONPATH": os.pathsep.join([str(CODE / "lpno" / "src"), os.environ.get("PYTHONPATH", "")])}
EXPECTED = list(csv.DictReader(open(HERE / "data" / "figure_files.csv")))


def run(script, *args, env=None):
    # one generator, in its own folder, writing into OUT
    r = subprocess.run([sys.executable, script, *args], cwd=FIGNB, env={**ENV, **(env or {})},
                       capture_output=True, text=True)
    status = "ok" if r.returncode == 0 else f"FAILED ({r.returncode})"
    print(f"{script:34s} {status}")
    if r.returncode:
        print(r.stdout[-1500:], r.stderr[-1500:])
    return r.returncode == 0


# The figure notebooks run on a kernel of this same interpreter: a kernel spec in a scratch JUPYTER_PATH, so the
# name "python3" cannot resolve to another installation.
KERNELS = OUT / "_kernel"
(KERNELS / "kernels" / "lpno").mkdir(parents=True, exist_ok=True)
(KERNELS / "kernels" / "lpno" / "kernel.json").write_text(json.dumps(
    {"argv": [sys.executable, "-m", "ipykernel_launcher", "-f", "{connection_file}"],
     "display_name": "lpno", "language": "python"}))


def run_notebook(name):
    r = subprocess.run([sys.executable, "-m", "jupyter", "nbconvert", "--to", "notebook", "--execute",
                        "--ExecutePreprocessor.kernel_name=lpno", "--ExecutePreprocessor.timeout=1200",
                        "--output-dir", str(OUT / "_executed"), name],
                       cwd=FIGNB, env={**ENV, "JUPYTER_PATH": str(KERNELS)}, capture_output=True, text=True)
    print(f"{name:34s} {'ok' if r.returncode == 0 else 'FAILED'}")
    if r.returncode:
        print(r.stderr[-2500:])
    return r.returncode == 0


import matplotlib
shown = OUT.relative_to(CODE) if OUT.is_relative_to(CODE) else OUT
print(f"figures written to {shown}; Python {sys.version.split()[0]}, matplotlib {matplotlib.__version__}")

figures written to figures_out; Python 3.10.18, matplotlib 3.10.6


## 1. Figures from committed data

Diagrams (`fig:pillars`, `fig:prov-chain`, `fig:prov-roles`, `fig:lineage`), the input channels (`fig:channels`), the
validation figures (`fig:calib-materials`, `fig:calib-xray` and Supplementary Note B), the acquisition rounds
(`fig:al-rounds`), the closing comparison (`fig:closing`), the datasets by melting regime (`fig:dataset-regimes`), and
the speed profiles and scale keys of the render figures.

In [2]:
for nb in ("provenance_diagram.ipynb",):
    run_notebook(nb)
for script in ("make_lineage_fig.py", "make_channel_fig.py",
               "make_calib_summary_parity.py", "make_calib_regime_panel.py", "make_calib_xray_overlays.py",
               "make_calib_keyhole_panels.py", "make_calib_transition_panels.py", "make_calib_cross_sections.py",
               "make_calib_history_panels.py",
               "make_al_rounds_fig.py", "make_closing_figs.py", "make_dataset_regimes_fig.py",
               "make_varspeed_speed_profiles.py", "make_render_keys.py"):
    run(script)

provenance_diagram.ipynb           ok


make_lineage_fig.py                ok


make_channel_fig.py                ok


make_calib_summary_parity.py       ok


make_calib_regime_panel.py         ok


make_calib_xray_overlays.py        ok


... (5 outputs left out)

make_closing_figs.py               ok


make_dataset_regimes_fig.py        ok


make_varspeed_speed_profiles.py    ok


make_render_keys.py                ok


## 2. Ground truth from the restricted record

* Figs 7-9 (`fig:rp-artifact`, `fig:varspeed-pred`, `fig:multibeam-pred`): `prepare_path_frames.py` reads the shown
  frame of each simulation from the payload tars and rebuilds the laser paths. `make_gaot_smoothing.py` draws
  `fig:gaot-smoothing` (Supplementary Note F) from the GAOT frame of Fig 7.
* Fig 12 (`fig:twoalloy-pred`): `prepare_truth_frames.py fig12` reads the joint RIGNO's evaluation previews from
  `restricted_files_main.tar`.
* Fig 10 (`fig:preheat-pred`): `prepare_truth_frames.py fig10` derives the three preheat simulations with aiida-pino's own
  derivation, in an AiiDA profile that holds the open archives and these three payloads:

```bash
verdi presto
verdi archive import lpno_provenance.aiida lpno_provenance_rounds.aiida lpno_provenance_evals.aiida
aiida-pino restore-payloads --deposit "$LPNO_RESTRICTED" --yes \
    --for c82581e6-33b6-46e3-8e6f-03b3163936e4 --for 6c188437-1f82-4895-9a90-1b799e3383c1 \
    --for e6ca3821-9589-41a4-b398-57e1628aac85
```

Each step is skipped when its frames are already present.

In [3]:
DATA = FIGNB / "data"
path_gt = sorted((DATA / "path_frames").glob("*_gt*.npz"))
if len(path_gt) >= 5:
    print(f"Figs 7-9 ground truth present ({len(path_gt)} frames)")
elif RESTRICTED:
    run("prepare_path_frames.py", env={"LPNO_RESTRICTED": RESTRICTED,
                                       "LPNO_FLOW3D_SCRIPTS": str(CODE / "flow3D_scripts" / "python")})
else:
    print("Figs 7-9: no ground-truth frames and no LPNO_RESTRICTED; these figures are skipped")

if (DATA / "twoalloy_truth_frames.npz").exists():
    print("Fig 12 ground truth present")
elif RESTRICTED:
    run("prepare_truth_frames.py", "fig12", env={"LPNO_RESTRICTED": RESTRICTED})
else:
    print("Fig 12: no ground truth and no LPNO_RESTRICTED; skipped")

if (DATA / "preheat_truth_frames.npz").exists():
    print("Fig 10 ground truth present")
elif os.environ.get("AIIDA_PATH") or Path("~/.aiida").expanduser().exists():
    run("prepare_truth_frames.py", "fig10")
else:
    print("Fig 10: no ground truth and no AiiDA profile with the restored payloads; skipped")

Figs 7-9 ground truth present (5 frames)
Fig 12 ground truth present
Fig 10 ground truth present


In [4]:
if len(sorted((DATA / "path_frames").glob("*_gt*.npz"))) >= 5:
    run("make_path_renders.py")
    run("make_gaot_smoothing.py")
if (DATA / "preheat_truth_frames.npz").exists():
    run_notebook("preheat_prediction_panels.ipynb")
if (DATA / "twoalloy_truth_frames.npz").exists():
    run("make_twoalloy_fig.py")
run("make_colorbars.py")          # the colour-bar strips, after the renders that also write them

make_path_renders.py               ok


make_gaot_smoothing.py             ok


preheat_prediction_panels.ipynb    ok


make_twoalloy_fig.py               ok


make_colorbars.py                  ok


True

## 3. What was produced

Every file the manuscript includes, whether it was written, and (with `LPNO_PUBLISHED_FIGS`) how it compares with the
published PDF. Pages are rasterized at 100 dpi. The published panels were drawn with matplotlib 3.3, 3.10 and 3.11, and these
place text glyphs and resample embedded rasters (the radiographs) differently by a pixel, so both pages are blurred over
2 pixels before the comparison: a page matches when under 1 % of its pixels then differ by more than 40 grey levels.
The raw, unblurred figure is listed too.

In [5]:
import numpy as np
from PIL import Image


def raster(pdf, tag):
    png = OUT / "_png" / f"{tag}_{pdf.stem}"
    png.parent.mkdir(exist_ok=True)
    subprocess.run(["pdftoppm", "-png", "-r", "100", "-singlefile", str(pdf), str(png)], check=True)
    return np.asarray(Image.open(f"{png}.png").convert("L"), int)


from PIL import ImageFilter


def differing(a, b, blur):
    # fraction of pixels differing by more than 40 grey levels, after a box blur of `blur` pixels
    h, w = min(a.shape[0], b.shape[0]), min(a.shape[1], b.shape[1])
    A, B = (Image.fromarray(x[:h, :w].astype(np.uint8)) for x in (a, b))
    if blur:
        A, B = A.filter(ImageFilter.BoxBlur(blur)), B.filter(ImageFilter.BoxBlur(blur))
    return float((np.abs(np.asarray(A, int) - np.asarray(B, int)) > 40).mean())


rows = []
for e in EXPECTED:
    new = OUT / e["file"]
    row = {"figure": e["figure"], "file": e["file"], "written": new.exists()}
    if PUBLISHED and new.exists() and (Path(PUBLISHED) / e["file"]).exists():
        a, b = raster(new, "new"), raster(Path(PUBLISHED) / e["file"], "pub")
        if a.shape != b.shape:
            row["size"] = f"{a.shape[1]}x{a.shape[0]} px against {b.shape[1]}x{b.shape[0]}"
        row["pixels differing (%)"] = round(100 * differing(a, b, 0), 3)
        row["after 2-px blur (%)"] = round(100 * differing(a, b, 2), 3)
        row["matches"] = row["after 2-px blur (%)"] < 1.0
    rows.append(row)
written = sum(r["written"] for r in rows)
print(f"{written} of {len(rows)} figure files written")
if PUBLISHED:
    print(f"{sum(r.get('matches', False) for r in rows)} match the published PDFs")
missing = sorted({r["figure"] for r in rows if not r["written"]})
print("figures not produced:", missing or "none")
for r in rows:
    if r["written"] and r.get("matches") is False:
        print(f"  over tolerance: {r['file']:34s} {r['after 2-px blur (%)']:.3f} % of pixels after the blur")
try:
    import pandas as pd
    display(pd.DataFrame(rows))
except ImportError:
    for r in rows:
        print(r)

81 of 81 figure files written
74 match the published PDFs
figures not produced: none
  over tolerance: rp_artifact_cbar.pdf               1.090 % of pixels after the blur
  over tolerance: multibeam_gt_3beam.pdf             1.701 % of pixels after the blur
  over tolerance: multibeam_pred_3beam.pdf           1.701 % of pixels after the blur
  over tolerance: multibeam_gt_4beam.pdf             1.701 % of pixels after the blur
  over tolerance: multibeam_pred_4beam.pdf           1.701 % of pixels after the blur
  over tolerance: multibeam_cbar.pdf                 1.090 % of pixels after the blur
  over tolerance: preheat3d_cbar.pdf                 6.342 % of pixels after the blur


,figure,file,written,pixels differing (%),after 2-px blur (%),matches,size
0,fig:pillars,three_pillars.pdf,True,1.391,0.507,True,NaN
1,fig:prov-chain,provenance_chain.pdf,True,0.681,0.114,True,NaN
2,fig:prov-roles,collection_roles.pdf,True,0.967,0.078,True,NaN
3,fig:calib-materials,calib_summary_parity.pdf,True,0.642,0.124,True,NaN
4,fig:calib-materials,calib_summary_regime.pdf,True,0.004,0.000,True,NaN
...,...,...,...,...,...,...,...
76,fig:calib-mesh,calib_mesh_c5.pdf,True,0.235,0.627,True,NaN
77,fig:lineage,lineage.pdf,True,0.010,0.000,True,NaN
78,fig:gaot-smoothing,gaot_smoothing_before.pdf,True,0.000,0.000,True,NaN
79,fig:gaot-smoothing,gaot_smoothing_after.pdf,True,0.000,0.000,True,NaN


With matplotlib 3.10, 74 of the 81 pages match. The other seven were drawn with other versions and differ in how text and
dashes are placed: `preheat3d_cbar` (3.3), and the colour bars and multi-beam panels of `fig:rp-artifact` and
`fig:multibeam-pred` (3.11). Regenerated with 3.11, those six are identical to the published files.
